# MirrorTopology Step 1 — Phase B・B-3-1 official-scale control battery（v0.1）
B-3-0 と同じ launcher 設計（外側 launcher lock → fresh scratch checkout → inventory が pins／script を束縛 → 環境 lock → script）。control 用 profile（**1 配置 × 2 系統**，N₀=10⁶・m=100・B=2000・N_fit=2×10⁵・B_KDE=2000）で negative・positive full predicate・conjunct-drop・brute-force・finite inventory・A5 cross-check を実行し，`B3_1_PASS` を最終 record で確定する。production の全 surviving-size 条件は主張しない。label は解放しない。所要時間の目安：Colab CPU で 15〜30 分。

In [ ]:
# --- 0. OUTER LAUNCHER LOCK (the only editable cell)
REPO_URL = 'https://github.com/tsujikeita/mirror-topology.git'
REPO_COMMIT = '<full 40-hex commit of the verification target>'
EXPECTED_INVENTORY_SHA256 = '<sha256 of engine/phaseB/B2_completion_inventory.json inside that commit>'
LAUNCHER_ID = 'MirrorTopology_Step1_B3_1_control_v0.1'


In [ ]:
# --- 1. fresh scratch checkout; clean tree; inventory bound; pins / script bound through the inventory
import subprocess, sys, os, json, hashlib, shutil, time, re
sha=lambda p: hashlib.sha256(open(p,'rb').read()).hexdigest()
assert re.fullmatch(r'[0-9a-f]{40}', REPO_COMMIT) and re.fullmatch(r'[0-9a-f]{64}', EXPECTED_INVENTORY_SHA256), 'launcher lock not filled'
RUN=f'/content/b3_1_runs/{time.strftime("%Y%m%dT%H%M%SZ", time.gmtime())}'; SCRATCH=f'{RUN}/scratch'; OUT=f'{RUN}/out'; os.makedirs(OUT)
subprocess.run(['git','clone','-q',REPO_URL,SCRATCH],check=True); subprocess.run(['git','-C',SCRATCH,'checkout','-q',REPO_COMMIT],check=True)
head=subprocess.check_output(['git','-C',SCRATCH,'rev-parse','HEAD']).decode().strip(); assert head==REPO_COMMIT, head
assert subprocess.check_output(['git','-C',SCRATCH,'status','--porcelain']).decode().strip()=='', 'scratch tree not clean'
MT=SCRATCH; PHASEB=f'{MT}/engine/phaseB'; INV=f'{PHASEB}/B2_completion_inventory.json'; inv_sha=sha(INV); assert inv_sha==EXPECTED_INVENTORY_SHA256, inv_sha
inv=json.load(open(INV)); PINS=f'{PHASEB}/b3/b3_1_pins.json'
assert sha(PINS)==inv['b3_sha256']['b3/b3_1_pins.json'] and sha(f'{PHASEB}/b3/b3_1_control.py')==inv['b3_sha256']['b3/b3_1_control.py']
pins=json.load(open(PINS)); assert 'repo' not in pins and pins['engine_version']==inv['engine_version'] and pins['schema']=='b3_1_pins_v1'
lock=dict(launcher_id=LAUNCHER_ID, repo_url=REPO_URL, commit=head, inventory_sha256=inv_sha, pins_sha256=sha(PINS), engine_version=inv['engine_version'], run_dir=RUN); json.dump(lock, open(f'{OUT}/launcher_lock.json','w'), indent=1); print(lock)


In [ ]:
# --- 2. environment lock
ex=pins['environment']
subprocess.run(['pip','install','-q',f"numpy=={ex['numpy']}",f"scipy=={ex['scipy']}",f"healpy=={ex['healpy']}",f"pot=={ex['pot']}",f"camb=={ex['camb']}",'threadpoolctl'],check=True)
os.environ['OPENBLAS_NUM_THREADS']='2'; os.environ['OMP_NUM_THREADS']='2'
import platform, numpy, scipy, healpy, ot, camb
live=dict(python=platform.python_version(), numpy=numpy.__version__, scipy=scipy.__version__, healpy=healpy.__version__, pot=ot.__version__, camb=camb.__version__)
mism={k:(live[k],ex[k]) for k in live if live[k]!=ex[k]}; assert not mism, f'environment lock failed: {mism}'; print('environment lock OK', live)


In [ ]:
# --- 3. control battery (fresh OUT/control; inventory-bound pins; required gates; evidence + run manifest)
CT=f'{OUT}/control'
rc=subprocess.run([sys.executable,f'{PHASEB}/b3/b3_1_control.py','--mt',MT,'--phaseb',PHASEB,'--out',CT,'--profile','control_official'],capture_output=True,text=True)
open(f'{OUT}/launcher_script_stdout.txt','w').write(rc.stdout); open(f'{OUT}/launcher_script_stderr.txt','w').write(rc.stderr); print(rc.stdout[-2000:])
rm=json.load(open(f'{CT}/b3_1_run_manifest.json')); script_ok=bool(rc.returncode==0 and rm.get('B3_1_PASS') is True); print('script rc', rc.returncode, 'B3_1_PASS', rm.get('B3_1_PASS'), rm.get('failures'))


In [ ]:
# --- 4. final record (composed; excludes itself from the inventory) + return list
def inventory(root, exclude=()):
    out={}
    for d,_,fs in os.walk(root):
        for f in fs:
            p=os.path.join(d,f); rel=os.path.relpath(p, root)
            if rel in exclude: continue
            out[rel]=dict(sha256=sha(p), bytes=os.path.getsize(p))
    return out
final=dict(launcher=lock, B3_1_PASS=bool(script_ok), stages=dict(checkout=True, environment=live, script_returncode=rc.returncode, script_pass=rm.get('B3_1_PASS'), script_gates=rm.get('gates'), script_failures=rm.get('failures'), timings=rm.get('timings')), execution_profile=rm.get('execution_profile'))
final['output_inventory']=inventory(OUT, exclude=('b3_1_final_record.json','b3_1_return_list.json'))
json.dump(final, open(f'{OUT}/b3_1_final_record.json','w'), indent=1); json.dump(dict(final_record_sha256=sha(f'{OUT}/b3_1_final_record.json'), files=final['output_inventory']), open(f'{OUT}/b3_1_return_list.json','w'), indent=1)
print(json.dumps({k:final[k] for k in ('B3_1_PASS',)}, indent=1), 'run dir:', RUN)


In [ ]:
# --- 5. (after the final record) zip RUN/out for the audit
import shutil
from google.colab import files
p = shutil.make_archive(f'/content/b3_1_out_{REPO_COMMIT[:12]}', 'zip', root_dir=OUT); print(p, os.path.getsize(p)); files.download(p)


## 監査へ渡すもの
`RUN/out/` 全体（launcher_lock・control/ の banks／plans seed0／3 結果 checkpoint／run manifest／log・final record・return list）と実行済み notebook。